In [1]:
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/merged_dataset
!rm -rf /content/test_dataset
!unzip -q "/content/drive/MyDrive/MRP/merged_dataset.zip" -d "/content/merged_dataset"
!unzip -q "/content/drive/MyDrive/MRP/test_dataset.zip" -d "/content/test_dataset"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64,
    validation_split=0.2,
    subset="training",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64,
    validation_split=0.2,
    subset="validation",
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/test_dataset/test_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64
)

Found 351500 files belonging to 2 classes.
Using 281200 files for training.
Found 351500 files belonging to 2 classes.
Using 70300 files for validation.
Found 18124 files belonging to 2 classes.


In [4]:
def fft_magnitude(image):
    # Convert to float32
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Compute FFT per channel
    fft = tf.signal.fft2d(tf.cast(image, tf.complex64))

    # Shift zero-frequency to center
    fft_shift = tf.signal.fftshift(fft)

    # Magnitude
    magnitude = tf.abs(fft_shift)

    # Log scale (critical for stability)
    magnitude = tf.math.log(magnitude + 1e-8)

    # Normalize to [0,1]
    magnitude = (magnitude - tf.reduce_min(magnitude)) / (tf.reduce_max(magnitude) - tf.reduce_min(magnitude) + 1e-8)

    return magnitude

In [5]:
train_fft = train_ds.map(lambda x, y: (fft_magnitude(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_fft = val_ds.map(lambda x, y: (fft_magnitude(x), y), num_parallel_calls=tf.data.AUTOTUNE)
test_fft = test_ds.map(lambda x, y: (fft_magnitude(x), y), num_parallel_calls=tf.data.AUTOTUNE)
train_fft = train_fft.prefetch(tf.data.AUTOTUNE)
val_fft = val_fft.prefetch(tf.data.AUTOTUNE)
test_fft = test_fft.prefetch(tf.data.AUTOTUNE)

In [6]:
def build_fft_cnn(input_shape=(256, 256, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 4
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Classification head
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu', name="fft_feature_vector"),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    return model

model = build_fft_cnn()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fft_feature_vector (Dense)      │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,361 (1.61 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [8]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_fft,
    validation_data=val_fft,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 870s 194ms/step - accuracy: 0.6708 - loss: 0.6374 - val_accuracy: 0.6742 - val_loss: 0.6326
Epoch 2/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 629s 143ms/step - accuracy: 0.6720 - loss: 0.6345 - val_accuracy: 0.6742 - val_loss: 0.6313
Epoch 3/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 642s 146ms/step - accuracy: 0.6720 - loss: 0.6341 - val_accuracy: 0.6742 - val_loss: 0.6314
Epoch 4/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 642s 146ms/step - accuracy: 0.6721 - loss: 0.6338 - val_accuracy: 0.6742 - val_loss: 0.6317


In [12]:
loss, acc = model.evaluate(val_fft)
print("Validation accuracy:", acc)

1099/1099 ━━━━━━━━━━━━━━━━━━━━ 57s 51ms/step - accuracy: 0.6742 - loss: 0.6313
Validation accuracy: 0.6742247343063354


In [13]:
loss, acc = model.evaluate(test_fft)
print("Validation accuracy:", acc)

284/284 ━━━━━━━━━━━━━━━━━━━━ 15s 50ms/step - accuracy: 0.6750 - loss: 0.6310
Validation accuracy: 0.6750165820121765


In [11]:
model.save("/content/drive/MyDrive/MRP/models/fft_cnn_baseline.h5")

In [ ]:
# Note: After run:
# model = load_model("/content/drive/MyDrive/MRP/models/fft_cnn_baseline.h5")
# feature_extractor = tf.keras.Model(
#     inputs=model.input,
#     outputs=model.get_layer("fft_feature_vector").output
# )